# Lab 08: WGAN on MNIST

**Goal:** Build a Wasserstein GAN and see the difference from BCE GAN.

What's new vs our previous GANs:
- **Critic** instead of Discriminator (no Sigmoid)
- **Wasserstein loss** instead of BCE (just means & subtraction)
- **Weight clipping** to enforce Lipschitz constraint
- **5:1 training** — Critic trains 5x per Generator update
- **RMSProp** instead of Adam
- **Meaningful loss curve** — the Wasserstein estimate (−C_loss) shrinks as images get better

**Expected compute** (approximate): main loop 25 epochs ≈ 15 min on Apple-silicon (`mps`), ≈ 5–10 min on a Colab T4, ≈ 25 min on CPU with the automatic 10k-image subset (~2.5 h on CPU with all 60k). The no-clipping experiment adds ~20%.

**How to use this lab:** each experiment has a **Predict** prompt — answer it before you run the cell.


In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available()  # Apple-silicon GPU
                      else "cpu")
print(f"Using: {device}")

## 1. Data

Same MNIST setup as before.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),   # scale to [-1, 1]
])

mnist = datasets.MNIST("data", train=True, download=True, transform=transform)

# Each epoch = 938 generator updates x (5 critic + 1 G) steps -- heavy on a CPU.
# CPU-only machines train on a 10k-image subset (None = all 60k images, used on GPU / mps).
TRAIN_SUBSET = 10_000 if device.type == "cpu" else None
if TRAIN_SUBSET:
    mnist = torch.utils.data.Subset(mnist, range(TRAIN_SUBSET))
loader = DataLoader(mnist, batch_size=64, shuffle=True)

## 2. Critic & Generator

Notice: the Critic has **no Sigmoid** at the end. It outputs a raw score.

Same shapes as Day 1's DCGAN, except z = 64 (as in the Day 2 slides) and no Sigmoid. BatchNorm in the critic is acceptable for weight-clipped WGAN (the original paper used it); WGAN-GP (Lab 09) will have to remove it.

In [ ]:
class Critic(nn.Module):
    """Scores images. Higher = more realistic. No Sigmoid."""
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            # 1 x 28 x 28
            nn.Conv2d(1, 64, 4, 2, 1),            # -> 64 x 14 x 14
            nn.LeakyReLU(0.2),

            nn.Conv2d(64, 128, 4, 2, 1),           # -> 128 x 7 x 7
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),

            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1),             # -> 1 score
            # NO SIGMOID!
        )

    def forward(self, x):
        return self.model(x)


class Generator(nn.Module):
    """Takes noise, produces 28x28 image."""
    def __init__(self, z_dim=64):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(z_dim, 256 * 7 * 7),
            nn.Unflatten(1, (256, 7, 7)),

            nn.ConvTranspose2d(256, 128, 4, 2, 1),  # -> 128 x 14 x 14
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.ConvTranspose2d(128, 1, 4, 2, 1),    # -> 1 x 28 x 28
            nn.Tanh(),
        )

    def forward(self, z):
        return self.model(z)

In [ ]:
z_dim = 64

critic = Critic().to(device)
gen    = Generator(z_dim).to(device)

print(f"Critic params:    {sum(p.numel() for p in critic.parameters()):,}")
print(f"Generator params: {sum(p.numel() for p in gen.parameters()):,}")

## 3. WGAN Training Loop

Key differences from BCE GAN:
1. **Loss = means & subtraction** (no BCELoss, no labels)
2. **Weight clipping** after each Critic update
3. **Train Critic 5 times** per Generator update
4. **RMSProp** optimizer

Simplification vs the paper: the paper draws a **fresh real batch** for every critic step; here the 5 critic steps reuse the same real batch (fresh fakes each time). This is common and keeps the loop readable.

**Predict:** which way will the printed `C_loss` move over 25 epochs — toward 0, or further from 0?

In [ ]:
# Hyperparameters
lr         = 5e-5
n_critic   = 5       # Critic updates per Generator update
clip_value = 0.01    # Weight clipping range
epochs     = 25

# RMSProp, NOT Adam
opt_C = torch.optim.RMSprop(critic.parameters(), lr=lr)
opt_G = torch.optim.RMSprop(gen.parameters(), lr=lr)

In [ ]:
# Track losses for plotting
history_C = []
history_G = []

# Fixed noise to watch progress
fixed_noise = torch.randn(16, z_dim, device=device)

for epoch in range(epochs):
    epoch_C_losses = []
    epoch_G_losses = []

    for real, _ in loader:
        real = real.to(device)
        bs = real.size(0)

        # ── Train Critic (5 times) ──
        for _ in range(n_critic):
            noise = torch.randn(bs, z_dim, device=device)
            fake  = gen(noise).detach()

            # WGAN Critic loss: push real scores up, fake scores down
            loss_C = -critic(real).mean() + critic(fake).mean()

            opt_C.zero_grad()
            loss_C.backward()
            opt_C.step()

            # Weight clipping — enforce Lipschitz
            for p in critic.parameters():
                p.data.clamp_(-clip_value, clip_value)

        # ── Train Generator (1 time) ──
        noise = torch.randn(bs, z_dim, device=device)
        fake  = gen(noise)

        # WGAN Generator loss: get high scores from Critic
        loss_G = -critic(fake).mean()

        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()

        epoch_C_losses.append(loss_C.item())
        epoch_G_losses.append(loss_G.item())

    avg_C = sum(epoch_C_losses) / len(epoch_C_losses)
    avg_G = sum(epoch_G_losses) / len(epoch_G_losses)
    history_C.append(avg_C)
    history_G.append(avg_G)

    if epoch % 5 == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch:3d} | C_loss: {avg_C:+.4f} | G_loss: {avg_G:+.4f}")

## 4. The Meaningful Loss Curve

Unlike BCE GAN, the WGAN loss **actually tracks image quality**.

`C_loss = −(mean C(real) − mean C(fake))`, so **−C_loss is the critic's estimate of the Wasserstein distance W**.
- First few hundred steps: the critic learns to separate real from fake, so −C_loss jumps **up** from 0.
- After that: as G improves the distributions get closer, so −C_loss **shrinks toward 0** (C_loss rises toward 0).

So: *shrinking W estimate = better images.* (G_loss on its own is not comparable across time — the critic keeps changing.)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot([-c for c in history_C], label="W estimate = -C_loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("W estimate")
ax1.set_title("Wasserstein estimate (should shrink as images improve)")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history_G, label="Generator loss", color="orange")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.set_title("Generator Loss")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Generated Images

In [ ]:
gen.eval()
with torch.no_grad():
    samples = gen(fixed_noise).cpu()

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    img = samples[i].squeeze()
    img = (img + 1) / 2   # [-1,1] -> [0,1]
    ax.imshow(img, cmap="gray")
    ax.axis("off")
plt.suptitle("WGAN Generated Digits", fontsize=14)
plt.tight_layout()
plt.show()
gen.train();

## 6. Experiment: What Happens Without Weight Clipping?

Let's see why the Lipschitz constraint matters.  
We'll train a mini WGAN for a few epochs WITHOUT clipping and watch the critic scores.

**Predict:** after 5 epochs, roughly how large will the real–fake gap be compared with the clipped critic's W estimate above (≈ 0.1–1)?

In [ ]:
# Fresh models for this experiment
critic_noclip = Critic().to(device)
gen_noclip    = Generator(z_dim).to(device)

opt_C2 = torch.optim.RMSprop(critic_noclip.parameters(), lr=5e-5)
opt_G2 = torch.optim.RMSprop(gen_noclip.parameters(), lr=5e-5)

print("Training WITHOUT weight clipping...")
print("Watch the Critic scores explode!\n")

for epoch in range(5):
    for i, (real, _) in enumerate(loader):
        real = real.to(device)
        bs = real.size(0)

        for _ in range(n_critic):
            noise = torch.randn(bs, z_dim, device=device)
            fake = gen_noclip(noise).detach()

            score_real = critic_noclip(real).mean()
            score_fake = critic_noclip(fake).mean()
            loss_C2 = -score_real + score_fake

            opt_C2.zero_grad()
            loss_C2.backward()
            opt_C2.step()
            # NO CLIPPING!

        noise = torch.randn(bs, z_dim, device=device)
        fake = gen_noclip(noise)
        loss_G2 = -critic_noclip(fake).mean()
        opt_G2.zero_grad()
        loss_G2.backward()
        opt_G2.step()

        if i % 200 == 0:
            print(f"Epoch {epoch} batch {i:4d} | "
                  f"C(real): {score_real.item():+10.2f} | "
                  f"C(fake): {score_fake.item():+10.2f} | "
                  f"Gap: {(score_real - score_fake).item():+10.2f}")

print("\nCompare the gap with the clipped critic's W estimate above.")
print("Without a Lipschitz constraint nothing bounds the scores: the critic can keep scaling them up,")
print("so the gap stops being a distance estimate.")

## 7. Experiment: Clip Value Sensitivity

First, where did the clipped weights end up?

**Predict:** what shape will the histogram of critic weights have with c = 0.01?

In [ ]:
# Look at our trained Critic's weight distribution
all_weights = []
for p in critic.parameters():
    all_weights.extend(p.data.cpu().flatten().tolist())

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(all_weights, bins=100, edgecolor="black", alpha=0.7)
ax.axvline(-clip_value, color="red", linestyle="--", label=f"Clip bounds ({-clip_value}, {clip_value})")
ax.axvline(clip_value, color="red", linestyle="--")
ax.set_xlabel("Weight value")
ax.set_ylabel("Count")
ax.set_title("Critic Weight Distribution After Training (clip=0.01)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Weights at -0.01: {sum(1 for w in all_weights if abs(w + 0.01) < 1e-6):,}")
print(f"Weights at +0.01: {sum(1 for w in all_weights if abs(w - 0.01) < 1e-6):,}")
print(f"Total weights:    {len(all_weights):,}")
print("\nNotice: many weights are pushed to the clip boundaries.")
print("This is the capacity loss problem that WGAN-GP (L09) will fix.")

### Your turn: clip value

Set `clip_value` to **0.1** and then **0.001** in the hyper-parameter cell (use `epochs = 5` to save time), then re-run in order: the model-creation cell (`critic = Critic()...`), the hyper-parameter cell (so the optimizers point at the new models), the training loop and the histogram.

**Predict first:** which value stops learning (vanishing gradients) and which goes unstable?

*Expected:* c = 0.001 → critic too weak, W estimate tiny, G barely improves; c = 0.1 → slow and unstable, scores and gradients grow large.

## Key Takeaways

1. **No Sigmoid, no BCELoss** — just raw scores and subtraction
2. **Weight clipping** keeps the Critic's Lipschitz constraint
3. **5:1 ratio** — stronger Critic = better gradient (opposite of BCE GAN!)
4. **Loss curve is meaningful** — −C_loss (the W estimate) shrinks as images improve
5. **Weight clipping has problems** — many weights hit the clip boundary

**Next: Lab 09 (WGAN-GP)** — replace weight clipping with gradient penalty for even better results.